In [3]:
# ============================================================
# FAST ONE-HOT RASTERIZATION OF SELECTED/ALL CATEGORIES EXCEPT LAND USE/WITH TOGGLE FOR SELECTED CLASSES
# OPTIMIZED VERSION: NO REPEATED MASKING
# ============================================================

import arcpy
import os
import re
import csv
import time
from datetime import datetime, timedelta
from arcpy.sa import ExtractByMask, Con, IsNull, SetNull, Raster

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

# ------------------------------------------------------------
# USER INPUTS
# ------------------------------------------------------------

mask_raster = r"E:\mask\Mask_land_Kalk_cog.tif"

input_shp = (
    r"N:\Prosjekter\411200_Kalk_i_grunnen\Berggrunn_N250_derivates"
    r"\BergFlate250_Hovedbergart.shp"
)

category_field = "Forenk_Ca"

out_dir = r"G:\Geology_rasters_2026\Bedrock"
os.makedirs(out_dir, exist_ok=True)

output_prefix = "Bedrock"

# ------------------------------------------------------------
# TOGGLE: SELECT ONLY SPECIFIC CLASSES?
# ------------------------------------------------------------

USE_SELECTED_CLASSES = True

# Write values exactly as they appear in category_field
SELECTED_CLASSES = ["Kalkstein, dolomittstein, mergelstein"]


local_tmp_dir = r"C:\temp\onehot_Bedrock"
os.makedirs(local_tmp_dir, exist_ok=True)

tmp_gdb = os.path.join(local_tmp_dir, "temp_onehot.gdb")

if arcpy.Exists(tmp_gdb):
    arcpy.management.Delete(tmp_gdb)

arcpy.management.CreateFileGDB(local_tmp_dir, "temp_onehot.gdb")

# ------------------------------------------------------------
# ENVIRONMENT FROM MASK
# ------------------------------------------------------------

mask_desc = arcpy.Describe(mask_raster)

arcpy.env.workspace = tmp_gdb
arcpy.env.scratchWorkspace = tmp_gdb
arcpy.env.snapRaster = mask_raster
arcpy.env.cellSize = mask_raster
arcpy.env.extent = mask_raster
arcpy.env.outputCoordinateSystem = mask_desc.spatialReference
arcpy.env.mask = mask_raster

cell_size = float(arcpy.management.GetRasterProperties(mask_raster, "CELLSIZEX").getOutput(0))

print("Mask raster:", mask_raster)
print("CRS:", mask_desc.spatialReference.name)
print("Cell size:", cell_size)
print("Rows:", arcpy.management.GetRasterProperties(mask_raster, "ROWCOUNT").getOutput(0))
print("Cols:", arcpy.management.GetRasterProperties(mask_raster, "COLUMNCOUNT").getOutput(0))

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def safe_name(value, max_len=70):
    value = str(value).strip()
    replacements = {
        "æ": "ae", "ø": "o", "å": "a",
        "Æ": "Ae", "Ø": "O", "Å": "A",
        "é": "e", "è": "e", "ê": "e",
        "É": "E", "È": "E", "Ê": "E",
        "'": "",
        '"': ""
    }
    for old, new in replacements.items():
        value = value.replace(old, new)

    value = re.sub(r"[^A-Za-z0-9_]+", "_", value)
    value = re.sub(r"_+", "_", value)
    value = value.strip("_")
    return value[:max_len] if value else "unknown"


def format_seconds(seconds):
    seconds = int(seconds)
    if seconds < 60:
        return f"{seconds} sec"
    minutes = seconds / 60
    if minutes < 60:
        return f"{minutes:.1f} min"
    return f"{minutes / 60:.2f} hr"


def progress_bar(current, total, width=30):
    fraction = current / total if total else 1
    filled = int(width * fraction)
    bar = "#" * filled + "-" * (width - filled)
    return f"[{bar}] {100 * fraction:5.1f}%"


def sql_value(v, field_type):
    if field_type in ["Integer", "SmallInteger", "Double", "Single"]:
        return str(v)
    else:
        return "'" + str(v).replace("'", "''") + "'"


# ------------------------------------------------------------
# CHECK INPUT
# ------------------------------------------------------------

if not arcpy.Exists(input_shp):
    raise FileNotFoundError(f"Input shapefile not found:\n{input_shp}")

fields = arcpy.ListFields(input_shp)
field_names = [f.name for f in fields]

if category_field not in field_names:
    print("\nAvailable fields:")
    for f in field_names:
        print(" -", f)
    raise ValueError(f"\nField '{category_field}' not found in:\n{input_shp}")

field_type = [f.type for f in fields if f.name == category_field][0]

# ------------------------------------------------------------
# PROJECT TO MASK CRS IF NEEDED
# ------------------------------------------------------------

in_desc = arcpy.Describe(input_shp)
projected_fc = os.path.join(tmp_gdb, f"{output_prefix}_projected")

if in_desc.spatialReference.name != mask_desc.spatialReference.name:
    print("\nProjecting polygon layer to mask CRS...")
    arcpy.management.Project(input_shp, projected_fc, mask_desc.spatialReference)
    work_fc = projected_fc
else:
    print("\nInput already has same CRS as mask.")
    work_fc = input_shp

# ------------------------------------------------------------
# COPY FEATURES LOCALLY
# ------------------------------------------------------------

local_fc = os.path.join(tmp_gdb, f"{output_prefix}_local")

print("\nCopying polygons to local temporary geodatabase...")
arcpy.management.CopyFeatures(work_fc, local_fc)

# ------------------------------------------------------------
# OPTIONAL: SELECT ONLY SPECIFIC CLASSES
# ------------------------------------------------------------

if USE_SELECTED_CLASSES:
    selected_fc = os.path.join(tmp_gdb, f"{output_prefix}_selected")

    values_sql = ",".join(sql_value(v, field_type) for v in SELECTED_CLASSES)
    field_sql = arcpy.AddFieldDelimiters(local_fc, category_field)
    where_clause = f"{field_sql} IN ({values_sql})"

    print("\nSelecting only requested classes:")
    print(where_clause)

    arcpy.management.MakeFeatureLayer(local_fc, "selected_layer", where_clause)
    arcpy.management.CopyFeatures("selected_layer", selected_fc)

    selected_count = int(arcpy.management.GetCount(selected_fc).getOutput(0))
    print("Selected polygons:", selected_count)

    if selected_count == 0:
        raise ValueError("No polygons selected. Check SELECTED_CLASSES and category_field values.")

    local_fc = selected_fc

# ------------------------------------------------------------
# UNIQUE CATEGORIES
# ------------------------------------------------------------

categories = sorted({
    row[0]
    for row in arcpy.da.SearchCursor(local_fc, [category_field])
    if row[0] not in [None, "", " "]
})

cat_to_id = {cat: i + 1 for i, cat in enumerate(categories)}

print(f"\nNumber of categories to rasterize in {category_field}: {len(categories)}")
for cat, class_id in cat_to_id.items():
    print(f" {class_id:02d} = {cat}")

# ------------------------------------------------------------
# LOOKUP CSV
# ------------------------------------------------------------

lookup_csv = os.path.join(out_dir, f"{output_prefix}_category_lookup.csv")

with open(lookup_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["class_id", "category", "output_raster"])

    for cat, class_id in cat_to_id.items():
        cat_safe = safe_name(cat)
        out_name = f"{output_prefix}_{class_id:02d}_{cat_safe}.tif"
        writer.writerow([class_id, cat, out_name])

print("\nSaved category lookup table:")
print(lookup_csv)

# ------------------------------------------------------------
# ADD CLASS ID FIELD
# ------------------------------------------------------------

class_field = "class_id"

if class_field not in [f.name for f in arcpy.ListFields(local_fc)]:
    arcpy.management.AddField(local_fc, class_field, "SHORT")

print("\nAssigning integer class IDs...")

with arcpy.da.UpdateCursor(local_fc, [category_field, class_field]) as cursor:
    for row in cursor:
        row[1] = cat_to_id.get(row[0], 0)
        cursor.updateRow(row)

# ------------------------------------------------------------
# RASTERIZE POLYGONS ONCE
# ------------------------------------------------------------

class_raster = os.path.join(tmp_gdb, f"{output_prefix}_class_raster")

print(f"\nRasterizing {output_prefix} polygons once using class_id...")
rasterize_start = time.time()

arcpy.conversion.PolygonToRaster(
    in_features=local_fc,
    value_field=class_field,
    out_rasterdataset=class_raster,
    cell_assignment="MAXIMUM_AREA",
    priority_field="NONE",
    cellsize=cell_size
)

print(f"Rasterization completed in {format_seconds(time.time() - rasterize_start)}")

# ------------------------------------------------------------
# MASK CLASS RASTER ONCE AND FILL NON-SELECTED LAND CELLS WITH 0
# ------------------------------------------------------------

print("\nMasking integer class raster once and filling non-selected land cells with 0...")
mask_start = time.time()

mask_r = Raster(mask_raster)

class_raster_masked = ExtractByMask(class_raster, mask_raster)

class_raster_full = SetNull(
    IsNull(mask_r),
    Con(IsNull(class_raster_masked), 0, class_raster_masked)
)

print(f"Masking completed in {format_seconds(time.time() - mask_start)}")

# ------------------------------------------------------------
# SAVE INTEGER CLASS COG
# ------------------------------------------------------------

class_raster_tif = os.path.join(out_dir, f"{output_prefix}_class_id.tif")

if arcpy.Exists(class_raster_tif):
    arcpy.management.Delete(class_raster_tif)

print("\nSaving integer class raster as COG:")
print(class_raster_tif)

save_start = time.time()

arcpy.management.CopyRaster(
    in_raster=class_raster_full,
    out_rasterdataset=class_raster_tif,
    pixel_type="16_BIT_UNSIGNED",
    nodata_value="65535",
    format="COG"
)

print(f"Integer class raster saved in {format_seconds(time.time() - save_start)}")

# ------------------------------------------------------------
# CREATE BINARY COG RASTERS
# ------------------------------------------------------------

print("\nCreating binary one-hot rasters...")
print("Binary rasters: 1 = selected class, 0 = other landmask pixels, NoData = outside landmask.")

total_maps = len(cat_to_id)
overall_start = time.time()
map_times = []

for idx, (cat, class_id) in enumerate(cat_to_id.items(), start=1):

    map_start = time.time()
    cat_safe = safe_name(cat)

    out_cog = os.path.join(
        out_dir,
        f"{output_prefix}_{class_id:02d}_{cat_safe}.tif"
    )

    print("\n" + "=" * 80)
    print(f"[{idx}/{total_maps}] {cat}")
    print(progress_bar(idx - 1, total_maps))
    print("=" * 80)

    if arcpy.Exists(out_cog):
        arcpy.management.Delete(out_cog)

    binary = SetNull(
        IsNull(mask_r),
        Con(class_raster_full == class_id, 1, 0)
    )

    arcpy.management.CopyRaster(
        in_raster=binary,
        out_rasterdataset=out_cog,
        pixel_type="8_BIT_UNSIGNED",
        nodata_value="255",
        format="COG"
    )

    elapsed_map = time.time() - map_start
    map_times.append(elapsed_map)

    completed = idx
    remaining = total_maps - completed
    avg_time = sum(map_times) / len(map_times)
    eta_sec = avg_time * remaining
    finish_time = datetime.now() + timedelta(seconds=eta_sec)

    print("\nSaved:", os.path.basename(out_cog))
    print(progress_bar(completed, total_maps))
    print(f"Map time:         {format_seconds(elapsed_map)}")
    print(f"Average/map:      {format_seconds(avg_time)}")
    print(f"ETA remaining:    {format_seconds(eta_sec)}")
    print(f"Estimated finish: {finish_time.strftime('%Y-%m-%d %H:%M:%S')}")

print("\nDone.")

Mask raster: E:\mask\Mask_land_Kalk_cog.tif
CRS: ETRS_1989_UTM_Zone_33N
Cell size: 10.0
Rows: 162232
Cols: 135236

Projecting polygon layer to mask CRS...

Copying polygons to local temporary geodatabase...

Selecting only requested classes:
Forenk_Ca IN ('Kalkstein, dolomittstein, mergelstein')
Selected polygons: 2119

Number of categories to rasterize in Forenk_Ca: 1
 01 = Kalkstein, dolomittstein, mergelstein

Saved category lookup table:
G:\Geology_rasters_2026\Bedrock\Bedrock_category_lookup.csv

Assigning integer class IDs...

Rasterizing Bedrock polygons once using class_id...
Rasterization completed in 1.2 min

Masking integer class raster once and filling non-selected land cells with 0...
Masking completed in 5.2 min

Saving integer class raster as COG:
G:\Geology_rasters_2026\Bedrock\Bedrock_class_id.tif
Integer class raster saved in 23.2 min

Creating binary one-hot rasters...
Binary rasters: 1 = selected class, 0 = other landmask pixels, NoData = outside landmask.

[1/1] Ka